In [1]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.cloud import storage
import vertexai
import numpy as np
from google.cloud.aiplatform.prediction import LocalModel

load_dotenv() 

I0000 00:00:1779675515.761036    4700 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


False

In [26]:
import os
from dotenv import load_dotenv

load_dotenv() 

PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

In [14]:
def build_model():
  model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=[10]),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
  ])

  return model

In [15]:
tf_model = build_model()

/home/ridwanfatur/work/portfolio/modular-ai/venvs/tensorflow/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
tf_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,929 (19.25 KB)

 Trainable params: 4,929 (19.25 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# example_input = np.random.randn(1, 10)
example_input = np.array([
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 9.0],
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 11.0],
])

In [18]:
tf_model(example_input)

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[-0.40251428],
       [-0.5284113 ]], dtype=float32)>

In [19]:
TENSORFLOW_ARTIFACT_DIR = "tensorflow"
TENSORFLOW_MODEL_NAME = "simple-model"
tf_model.export(f"{TENSORFLOW_ARTIFACT_DIR}/{TENSORFLOW_MODEL_NAME}")

INFO:tensorflow:Assets written to: tensorflow/simple-model/assets


INFO:tensorflow:Assets written to: tensorflow/simple-model/assets


Saved artifact at 'tensorflow/simple-model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 10), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140452545409296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140452545411024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140452716897552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140452545407952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140452545409488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140452545410832: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [20]:
DOCKER_IMAGE_NAME = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-13:latest"

In [23]:
deployed_local_model = LocalModel(
    serving_container_image_uri=DOCKER_IMAGE_NAME,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_environment_variables={
        "model_path": "/home/ridwanfatur/work/portfolio/modular-ai/tensorflow/tensorflow/simple-model",
        "model_name": "simple-model"
        # "model_path": f"/{TENSORFLOW_ARTIFACT_DIR}/{TENSORFLOW_MODEL_NAME}",
        # "model_name": TENSORFLOW_MODEL_NAME                         
    }
)

In [22]:
!pwd

/home/ridwanfatur/work/portfolio/modular-ai/deployments/vertex-ai/tensorflow


In [24]:
deployed_local_model.get_serving_container_spec()

image_uri: "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-13:latest"
env {
  name: "model_path"
  value: "/home/ridwanfatur/work/portfolio/modular-ai/tensorflow/tensorflow/simple-model"
}
env {
  name: "model_name"
  value: "simple-model"
}
predict_route: "/predict"
health_route: "/health"

In [25]:
local_endpoint = deployed_local_model.deploy_to_local_endpoint(
    artifact_uri=f"{TENSORFLOW_ARTIFACT_DIR}/{TENSORFLOW_MODEL_NAME}",
)

In [27]:
local_endpoint.serve()

ERROR:google.cloud.aiplatform.prediction.local_endpoint:Exception during starting serving: ('The health check never succeeded.', '', 1).


DockerError: ('The health check never succeeded.', '', 1)

In [ ]:
local_endpoint.print_container_logs(show_all=True)

In [ ]:
instances = [
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 9.0],
    [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 11.0],
]

response = local_endpoint.predict(request={"instances": instances})
print(response)

In [ ]:
DOCKER_IMAGE_NAME

In [ ]:
# !docker ps -a | grep tf2-cpu.2-13
# !docker ps -a | grep tf2-cpu.2-13 | awk '{print $1}' | xargs docker stop 2>/dev/null
# !docker ps -a | grep tf2-cpu.2-13 | awk '{print $1}' | xargs docker rm